### Consumo básico de un LLM local

Acá no hay API, no hay key, no hay costo por petición — `ollama.chat()` habla directo con el modelo que corre en tu propia máquina (servicio Ollama en background). Es el primer contacto con un LLM real: mandás un mensaje con rol `user`, el modelo responde con rol `assistant`. Mismo patrón `messages=[...]` que vas a ver en cualquier API cloud (OpenAI, Anthropic, Gemini) — acá lo practicás gratis y sin conexión a internet.

In [7]:
import ollama

response = ollama.chat(model="llama3.2", messages=[
    {"role": "user", "content": "¿Qué es un token en el contexto de un LLM? Respondé en 2 líneas."}
])
print(response["message"]["content"])

En el contexto de un modelo de lenguaje grande (LLM), un token es una unidad de información que representa una parte del lenguaje, como una palabra, un símbolo o un carácter. Estos tokens se utilizan para entrenar y evaluar al modelo, y se pueden agrupar para formar entidades más grandes, como frases o oraciones.


### Token ≠ palabra

`prompt_eval_count` y `eval_count` son los contadores reales que usa el modelo — no cuentan palabras, cuentan tokens (fragmentos de texto después de aplicar BPE). Esto importa porque en las APIs pagas facturás por token, no por palabra, y porque el límite de contexto también se mide en tokens. Ver el número acá, antes de la teoría, ayuda a que el concepto no quede abstracto.

In [8]:
print("Tokens del prompt:", response["prompt_eval_count"])
print("Tokens de la respuesta:", response["eval_count"])


Tokens del prompt: 46
Tokens de la respuesta: 82


## Tokens: qué son de verdad

Un token no es una palabra. Es una pieza que sale de partir el texto con **BPE** (Byte Pair Encoding): las palabras comunes quedan enteras, las raras se parten en pedazos más chicos y reusables. Por eso `prompt_eval_count` no coincide con contar palabras.

In [1]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
texto = "La tokenización BPE parte palabras infrecuentes en subpalabras, como inteligibilidad."
ids = enc.encode(texto)
piezas = [enc.decode([i]) for i in ids]

print(f"Texto: {texto!r}")
print(f"Cantidad de tokens: {len(ids)}")
print("Piezas:", piezas)


Texto: 'La tokenización BPE parte palabras infrecuentes en subpalabras, como inteligibilidad.'
Cantidad de tokens: 19
Piezas: ['La', ' token', 'ización', ' B', 'PE', ' parte', ' palabras', ' inf', 'rec', 'uentes', ' en', ' sub', 'pal', 'abras', ',', ' como', ' intelig', 'ibilidad', '.']


## Contexto: el límite de lo que el modelo puede "ver"

La ventana de contexto es cuántos tokens entran de una sola vez (prompt + respuesta). Si el prompt se pasa, el runtime corta tokens en silencio — no avisa, no tira error. Ollama define esto con `num_ctx` (default bajo, 2048). Forzamos un `num_ctx` chico para ver el corte en acción: metemos un dato al principio de un texto largo y preguntamos por él.

In [25]:
import ollama

prompt_largo = (
    "Mi color favorito es AZUL-7. "
    + "Esto es relleno para ocupar espacio y empujar el dato mencionado antes fuera de la ventana de contexto. " * 15
    + "¿Cuál dije que era mi color favorito?"
)

for ctx in (64, 2048):
    response = ollama.chat(
        model="llama3.2",
        messages=[{"role": "user", "content": prompt_largo}],
        options={"num_ctx": ctx},
    )
    print(f"--- num_ctx={ctx} ---")
    print("Tokens de entrada:", response["prompt_eval_count"])
    print("Respuesta:", response["message"]["content"])
    print()

--- num_ctx=64 ---
Tokens de entrada: 34
Respuesta: No dije nada sobre el color de tu favorito. Esto es el inicio de nuestra conversación, por lo que no tengo ningún dato anterior para referirme. ¿En qué puedo ayudarte hoy?

--- num_ctx=2048 ---
Tokens de entrada: 437
Respuesta: No mencionaste que AZUL-7 era tu color favorito. De hecho, solo mencionaste que era "AZUL-7" para ocupar espacio y empujar el dato mencionado antes fuera de la ventana de contexto, lo cual parece ser un ejercicio de relleno para demostrar un punto. No hay una afirmación explícita sobre tu color favorito.



**Qué muestra esto:** con `num_ctx=64` el modelo solo procesó 34 tokens (el prompt real mide ~412 con tiktoken) — el dato del principio quedó afuera de la ventana, cortado en silencio, y el modelo lo admite ("no tengo acceso a la conversación anterior"). Con `num_ctx=2048` entra el prompt casi completo (437 tokens con overhead del chat template), pero igual falla en encontrar el dato: eso ya no es límite de contexto, es límite de capacidad de recall de un modelo chico (3B) en textos con mucho relleno repetido — el problema clásico de "needle in a haystack". Contexto grande no es lo mismo que memoria perfecta.

### Manejo seguro de credenciales

La API key nunca va hardcodeada en el código ni en el notebook — vive en `.env` (archivo local, ignorado por git) y se carga en runtime con `load_dotenv()`. Acá solo se verifica que la key exista y su longitud, nunca se imprime la key completa. Es la práctica mínima para no filtrar credenciales por accidente al compartir código o subirlo a un repo.

In [16]:
from dotenv import load_dotenv
import os

load_dotenv()
key = os.getenv("GOOGLE_API_KEY")
print("Key cargada:", key is not None, "- longitud:", len(key) if key else 0)

Key cargada: True - longitud: 53


### Modelo local vs modelo frontier

Mismo prompt que la celda 1, pero ahora contra Gemini (API cloud de Google) en vez de llama3.2 local. Deja comparar dos cosas a la vez: la sintaxis distinta de cada SDK (`ollama.chat` vs `client.models.generate_content`), y la diferencia de calidad/precisión entre un modelo chico que corre en tu CPU y un modelo grande que corre en la infraestructura de Google.

In [20]:
from google import genai

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

response = client.models.generate_content(
    model="gemini-flash-lite-latest",
    contents="¿Qué es un token en el contexto de un LLM? Respondé en 2 líneas."
)
print(response.text)

Un token es un fragmento de texto (una palabra, parte de ella o un signo de puntuación) que los modelos de lenguaje utilizan como unidad básica para leer y generar información.


### Preprocesamiento de datos para un LLM

Un LLM no necesita ni quiere HTML crudo — scripts, estilos, navegación y demás son ruido que gasta tokens sin aportar información útil. `BeautifulSoup` limpia el HTML y deja solo el texto legible, que después se usa como contexto para el modelo. Es el primer paso del Proyecto 1 (generador de folletos): sin este filtro, el prompt se llenaría de basura y el resultado sería peor.

In [18]:
import requests
from bs4 import BeautifulSoup

url = "https://facebook.com"  # cambiá por el sitio que quieras usar de ejemplo
headers = {"User-Agent": "Mozilla/5.0"}

resp = requests.get(url, headers=headers)
soup = BeautifulSoup(resp.content, "html.parser")

for tag in soup(["script", "style", "img", "input", "nav", "header", "footer"]):
    tag.decompose()

texto = soup.get_text(separator="\n", strip=True)
# saca líneas vacías y duplicadas consecutivas
lineas = [l for l in texto.split("\n") if l.strip()]
texto_limpio = "\n".join(lineas)
#print(texto_limpio[:1500])

system_prompt = "Sos un redactor de marketing. A partir del texto de una empresa, escribís un folleto breve y atractivo en español, con título, 2-3 párrafos y un cierre con llamado a la acción."

response = client.models.generate_content(
    model="gemini-flash-lite-latest",
    contents=f"{system_prompt}\n\nTexto de la empresa:\n{texto_limpio[:4000]}"
)
print(response.text)

¡Hola! Como redactor de marketing, he transformado el texto técnico y de inicio de sesión de Facebook en un folleto comercial dinámico y persuasivo. Aquí lo tenés:

***

# Conéctate con lo que más te importa: Tu mundo, en un solo lugar

¿Querés saber qué está pasando con tus amigos, descubrir comunidades que comparten tus mismas pasiones y explorar todo aquello que te inspira? Facebook es mucho más que una red social; es el punto de encuentro digital donde tus intereses cobran vida, los recuerdos se atesoran y las conversaciones nunca se detienen.

Descubrí un universo de entretenimiento y conexión diseñado a tu medida. Desde videos atrapantes y grupos exclusivos hasta herramientas innovadoras como Meta AI y experiencias inmersivas con Meta Quest y Ray-Ban Meta, te ofrecemos tecnología de punta para potenciar tu día a día. Además, mantenete cerca de los tuyos al instante a través de Messenger e Instagram, sin perderte ningún detalle.

Es el momento de unirte a millones de personas que 